In [1]:
!pip install transformers torch datasets seqeval

import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from datasets import load_dataset

# Load your dataset directly from Hugging Face
hf_dataset = load_dataset("Fatimasajid/code-switching-codesaviours-si26-fatima")

# Convert to pandas
df = hf_dataset['train'].to_pandas()

print(df.head())
print(df.columns)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=cb20bb6bcc5764550a7e26ffdd1f820ec902711f666929a291bf5387f309aba4
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


README.md:   0%|          | 0.00/826 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


dataset.csv:   0%|          | 0.00/68.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1358 [00:00<?, ? examples/s]

                                            sentence    word label
0  Aaj office mein bohot kaam tha, I couldn't eve...     Aaj   URD
1  Aaj office mein bohot kaam tha, I couldn't eve...  office   URD
2  Aaj office mein bohot kaam tha, I couldn't eve...    mein   URD
3  Aaj office mein bohot kaam tha, I couldn't eve...   bohot   URD
4  Aaj office mein bohot kaam tha, I couldn't eve...    kaam   URD
Index(['sentence', 'word', 'label'], dtype='object')


In [2]:
!pip install transformers torch datasets seqeval

from datasets import load_dataset
import pandas as pd

hf_dataset = load_dataset("Fatimasajid/code-switching-codesaviours-si26-fatima")
df = hf_dataset['train'].to_pandas()

print(df.shape)
print(df.columns)
print(df.head())

Repo card metadata block was not found. Setting CardData to empty.


(1358, 3)
Index(['sentence', 'word', 'label'], dtype='object')
                                            sentence    word label
0  Aaj office mein bohot kaam tha, I couldn't eve...     Aaj   URD
1  Aaj office mein bohot kaam tha, I couldn't eve...  office   URD
2  Aaj office mein bohot kaam tha, I couldn't eve...    mein   URD
3  Aaj office mein bohot kaam tha, I couldn't eve...   bohot   URD
4  Aaj office mein bohot kaam tha, I couldn't eve...    kaam   URD


In [3]:
from sklearn.model_selection import train_test_split

label2id = {'URD': 0, 'ENG': 1, 'MIX': 2}
id2label = {0: 'URD', 1: 'ENG', 2: 'MIX'}

sentences = df.groupby('sentence').apply(
    lambda x: (x['word'].tolist(), x['label'].tolist())
).tolist()

train_data, test_data = train_test_split(sentences, test_size=0.2, random_state=42)
print(f"Training sentences: {len(train_data)}")
print(f"Testing sentences: {len(test_data)}")

Training sentences: 160
Testing sentences: 40


/tmp/ipykernel_855/2468429731.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sentences = df.groupby('sentence').apply(


In [4]:
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                           TrainingArguments, Trainer, DataCollatorForTokenClassification)
from datasets import Dataset

model_name = 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(examples['words'], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples['labels']):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []
        prev_word = None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word:
                label_ids.append(label2id[label[word_id]])
            else:
                label_ids.append(-100)
            prev_word = word_id
        labels.append(label_ids)
    tokenized['labels'] = labels
    return tokenized

def to_hf_dataset(data):
    return Dataset.from_dict({
        'words': [d[0] for d in data],
        'labels': [d[1] for d in data]
    })

train_ds = to_hf_dataset(train_data).map(tokenize_and_align_labels, batched=True)
test_ds = to_hf_dataset(test_data).map(tokenize_and_align_labels, batched=True)

print("Tokenization done!")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

Tokenization done!


In [7]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer)
)

print("Starting training...")
trainer.train()
print("Training complete!")

Starting training...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.759931,0.519551
2,0.400796,0.244775
3,0.258536,0.225828
4,0.221367,0.206508
5,0.188714,0.198073


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete!


In [8]:
!pip install seqeval

from seqeval.metrics import classification_report
import numpy as np

predictions, labels, _ = trainer.predict(test_ds)
predictions = np.argmax(predictions, axis=2)

true_predictions = [
    [id2label[p] for (p, l) in zip(pred, label) if l != -100]
    for pred, label in zip(predictions, labels)
]
true_labels = [
    [id2label[l] for (p, l) in zip(pred, label) if l != -100]
    for pred, label in zip(predictions, labels)
]

print(classification_report(true_labels, true_predictions))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

          NG       0.89      0.96      0.92       107
          RD       0.87      0.83      0.85        92

   micro avg       0.88      0.90      0.89       199
   macro avg       0.88      0.89      0.89       199
weighted avg       0.88      0.90      0.89       199



/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: URD seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ENG seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


In [9]:
from sklearn.metrics import classification_report

# Flatten the lists
flat_true = [label for sent in true_labels for label in sent]
flat_pred = [label for sent in true_predictions for label in sent]

print(classification_report(flat_true, flat_pred, digits=2))

              precision    recall  f1-score   support

         ENG       0.89      0.96      0.92       107
         URD       0.98      0.93      0.95       179

    accuracy                           0.94       286
   macro avg       0.93      0.94      0.94       286
weighted avg       0.94      0.94      0.94       286



In [10]:
from huggingface_hub import notebook_login
notebook_login()  # paste your HuggingFace token when prompted

In [11]:
repo_name = "language-id-codesaviours-si26-fatima"

model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

print(f"Model published at: https://huggingface.co/Fatimasajid/{repo_name}")

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...zvj9ypw/model.safetensors:   0%|          | 13.6kB / 1.11GB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpga1f3gp3/tokenizer.json: 100%|##########| 17.1MB / 17.1MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Model published at: https://huggingface.co/Fatimasajid/language-id-codesaviours-si26-fatima
